# SIT330 2.3HD Notebook  
## Robust debiasing for toxicity detection under demographic subpopulation shift and dialectal variation

This notebook is designed as the experimental companion for the HD research-paper task. It continues the accepted Distinction research problem: testing whether debiasing methods for toxicity detection remain effective under demographic subpopulation shift and whether the same fairness gains transfer to dialectal variation.

The notebook implements a controlled experimental workflow:

1. **CivilComments-WILDS experiment** for demographic subpopulation shift.
2. **ERM baseline** as the non-debiased reference point.
3. **Counterfactual Data Augmentation (CDA)** as a simple debiasing baseline.
4. **Counterfactual Logit Pairing (CLP)** as a stronger counterfactual debiasing baseline.
5. **Proposed method: Group-Balanced CLP (GB-CLP)**, which combines counterfactual consistency with group-balanced training.
6. **Fairness-aware evaluation**, including worst-group accuracy, Subgroup AUC, BPSN AUC, BNSP AUC, and false-positive-rate gaps.
7. **Dialectal transfer evaluation** using the Davidson hate/offensive-language dataset, with support for official dialect-probability files or a clearly labelled exploratory proxy.
8. **Error analysis and counterfactual sensitivity analysis** for the final paper.

## Runtime recommendation

For the final HD experiment, **Google Colab with GPU is strongly recommended**. The notebook fine-tunes transformer models and may train several methods. A local computer can be used only for smoke-testing with `QUICK_MODE = True`.

Recommended setup:

- **Smoke test:** local computer or Colab CPU/GPU, `QUICK_MODE = True`, small samples, 1 epoch.
- **Final evidence:** Colab GPU, `QUICK_MODE = False`, 3 seeds if time allows.
- **Best option:** Colab Pro/Pro+ or any environment with NVIDIA GPU and at least 12GB VRAM.

The Distinction task motivated CivilComments-WILDS and dialectal evaluation. The HD task requires implementation, controlled experiments, critical analysis, and a full ACM-style research paper, so do not rely only on quick-mode results.

In [ ]:
# ============================================================
# 0. Optional dependency installation
# ============================================================
# In Google Colab, this cell installs the required libraries.
# Locally, you can run the same command in your terminal:
#   pip install -U datasets transformers accelerate evaluate scikit-learn pandas numpy tqdm matplotlib

!pip install -U wilds
!pip install -q --force-reinstall "numpy==2.0.2" "pandas==2.2.2"
!pip install -q wilds datasets transformers evaluate accelerate scikit-learn matplotlib tqdm
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = [
    "datasets",
    "transformers",
    "accelerate",
    "evaluate",
    "sklearn",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
]

def package_available(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None

missing = [p for p in REQUIRED_PACKAGES if not package_available(p)]
IN_COLAB = "google.colab" in sys.modules

if missing:
    print("Missing packages:", missing)
    if IN_COLAB:
        print("Installing packages for Colab...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "datasets", "transformers", "accelerate", "evaluate",
            "scikit-learn", "pandas", "numpy", "tqdm", "matplotlib"
        ])
    else:
        print("Local environment detected. Install missing packages manually if this cell cannot install them.")
else:
    print("All required packages appear to be available.")

In [ ]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

import os
import re
import gc
import math
import random
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

from tqdm.auto import tqdm
from IPython.display import display

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 120)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

# QUICK_MODE is for debugging the notebook end-to-end.
# For final run, set QUICK_MODE = False and run on a GPU.
QUICK_MODE = False

# Strong but manageable transformer baseline.
MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 192
SEED = 42

# Methods:
#   erm       = standard empirical risk minimisation baseline
#   cda       = counterfactual data augmentation
#   clp       = counterfactual logit pairing
#   proposed  = proposed Group-Balanced CLP (GB-CLP)
METHODS_TO_RUN = ["erm", "cda", "clp", "proposed"]

if QUICK_MODE:
    N_TRAIN = 2500
    N_VAL = 1000
    N_TEST = 1000
    N_DIALECT = 1500
    EPOCHS = 1
    BATCH_SIZE = 16
else:
    # Full mode can be expensive. Use Colab GPU..
    N_TRAIN = 50000
    N_VAL = 10000
    N_TEST = 20000
    N_DIALECT = 10000
    EPOCHS = 10
    BATCH_SIZE = 64

LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
CLP_LAMBDA = 1.0
USE_AMP = True
NUM_WORKERS = 4
PIN_MEMORY = True
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_MIN_DELTA = 1e-4
BEST_MODEL_METRIC = "val_roc_auc"

OUTPUT_DIR = Path("outputs_hd_toxicity_debiasing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "QUICK_MODE": QUICK_MODE,
    "MODEL_NAME": MODEL_NAME,
    "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "OUTPUT_DIR": str(OUTPUT_DIR),
})

## Proposed method: Group-Balanced Counterfactual Logit Pairing

The proposed method is **Group-Balanced CLP (GB-CLP)**.

The motivation is direct: if the limitation is that fairness gains may not survive demographic subpopulation shift, then the training objective should not only make identity-swapped examples consistent, but should also reduce domination by majority groups.

For a training example $(x_i, y_i)$ and its identity-swapped counterfactual $x_i^{cf}$, the model produces logits $z_i = f_\theta(x_i)$ and $z_i^{cf} = f_\theta(x_i^{cf})$. The proposed loss is

$$
\mathcal{L}_{GB\text{-}CLP}
=
w_{g_i}
\left[
\mathrm{BCE}(z_i, y_i)
+
\mathrm{BCE}(z_i^{cf}, y_i)
\right]
+
\lambda
\left\lVert z_i - z_i^{cf} \right\rVert_2^2 ,
$$

where $g_i$ is the demographic group indicator and $w_{g_i}$ is an inverse-frequency group weight normalised to have mean 1. The BCE terms keep toxicity prediction accurate, while the CLP term discourages prediction changes caused only by identity-term swaps. The group weighting is the part that makes the method more explicitly targeted at subpopulation shift.

In [ ]:
# ============================================================
# 3. Load CivilComments-WILDS using the official WILDS package
# ============================================================
from wilds import get_dataset
from tqdm.auto import tqdm

WILDS_ROOT = "./wilds_data"

wilds_civil = get_dataset(
    dataset="civilcomments",
    download=True,
    root_dir=WILDS_ROOT
)

print("Dataset:", wilds_civil)
print("Metadata fields:", wilds_civil.metadata_fields)
print("Split dictionary:", wilds_civil.split_dict)

civil_subsets = {
    "train": wilds_civil.get_subset("train", transform=None),
    "validation": wilds_civil.get_subset("val", transform=None),
    "test": wilds_civil.get_subset("test", transform=None),
}

for name, subset in civil_subsets.items():
    print(name, len(subset))

In [ ]:
# ============================================================
# 4. CivilComments preprocessing
# ============================================================

IDENTITY_COLS = [
    "male", "female", "LGBTQ", "christian", "muslim",
    "other_religions", "black", "white"
]

def subset_to_df(subset, split_name: str) -> pd.DataFrame:
    rows = []
    metadata_fields = list(wilds_civil.metadata_fields)

    for i in tqdm(range(len(subset)), desc=f"Converting {split_name}"):
        x, y, metadata = subset[i]

        row = {
            "text": str(x),
            "label": int(y),
            "split_name": split_name,
        }

        metadata_values = metadata.tolist()
        for field, value in zip(metadata_fields, metadata_values):
            row[field] = int(value)

        rows.append(row)

    df = pd.DataFrame(rows)

    available_identity_cols = [c for c in IDENTITY_COLS if c in df.columns]

    for c in available_identity_cols:
        df[c] = df[c].fillna(0).astype(int)

    if "identity_any" not in df.columns:
        df["identity_any"] = (df[available_identity_cols].sum(axis=1) > 0).astype(int)
    else:
        df["identity_any"] = df["identity_any"].fillna(0).astype(int)

    def primary_identity(row):
        active = [c for c in available_identity_cols if int(row[c]) == 1]
        if len(active) == 0:
            return "no_identity"
        return active[0]

    df["primary_identity"] = df.apply(primary_identity, axis=1)
    df["wg_group"] = "y" + df["label"].astype(str) + "_id" + df["identity_any"].astype(str)

    keep_cols = [
        "text", "label", "identity_any", "primary_identity",
        "wg_group", "split_name"
    ] + available_identity_cols

    keep_cols = list(dict.fromkeys(keep_cols))

    return df[keep_cols].dropna(subset=["text", "label"]).reset_index(drop=True)


civil_train = subset_to_df(civil_subsets["train"], "train")
civil_val = subset_to_df(civil_subsets["validation"], "validation")
civil_test = subset_to_df(civil_subsets["test"], "test")

In [ ]:
def stratified_sample(df: pd.DataFrame, n: Optional[int], seed: int = 42) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df.sample(frac=1, random_state=seed).reset_index(drop=True)

    group_cols = ["label", "identity_any"]

    sampled_parts = []

    for _, group_df in df.groupby(group_cols):
        group_n = int(round(n * len(group_df) / len(df)))
        group_n = max(1, group_n)

        sampled_parts.append(
            group_df.sample(
                n=min(group_n, len(group_df)),
                random_state=seed,
                replace=False
            )
        )

    sampled = pd.concat(sampled_parts, ignore_index=True)

    if len(sampled) > n:
        sampled = sampled.sample(n=n, random_state=seed).reset_index(drop=True)

    elif len(sampled) < n:
        missing = n - len(sampled)

        remaining = df.drop(index=sampled.index, errors="ignore")

        if len(remaining) >= missing:
            extra = remaining.sample(n=missing, random_state=seed, replace=False)
        else:
            extra = df.sample(n=missing, random_state=seed, replace=True)

        sampled = pd.concat([sampled, extra], ignore_index=True)

    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

civil_train = stratified_sample(civil_train, N_TRAIN, SEED)
civil_val = stratified_sample(civil_val, N_VAL, SEED)
civil_test = stratified_sample(civil_test, N_TEST, SEED)

print("Train:", civil_train.shape)
print("Val:", civil_val.shape)
print("Test:", civil_test.shape)

print("Columns:", civil_train.columns.tolist())

display(civil_train.head())

display(
    civil_train.groupby(["label", "identity_any"])
    .size()
    .rename("count")
    .reset_index()
)

In [ ]:
# ============================================================
# 5. Counterfactual identity-term swapping
# ============================================================

# These identity swaps are intentionally simple and transparent.
# Important limitation:
# Identity swapping can create unnatural or socially simplified text.
# This should be reported in the paper as a limitation of counterfactual augmentation / CLP.

IDENTITY_SWAP_PAIRS = [
    ("muslim", "christian"),
    ("islam", "christianity"),
    ("black", "white"),
    ("woman", "man"),
    ("women", "men"),
    ("female", "male"),
    ("gay", "straight"),
    ("lesbian", "heterosexual"),
    ("transgender", "cisgender"),
]

def validate_swap_pairs(pairs: List[Tuple[str, str]]) -> None:
    """
    Checks that no identity term appears in more than one pair.
    This avoids uncontrolled chain-swapping.
    """
    all_terms = []
    for a, b in pairs:
        all_terms.extend([a.lower(), b.lower()])

    duplicated = sorted({t for t in all_terms if all_terms.count(t) > 1})

    if duplicated:
        raise ValueError(
            f"Duplicate identity terms found in swap pairs: {duplicated}. "
            "Each term should appear in only one pair to avoid chain swapping."
        )

validate_swap_pairs(IDENTITY_SWAP_PAIRS)


def match_case(replacement: str, original: str) -> str:
    """
    Light case preservation.
    Example:
    muslim -> christian
    Muslim -> Christian
    MUSLIM -> CHRISTIAN
    """
    if original.isupper():
        return replacement.upper()
    if original[:1].isupper():
        return replacement.capitalize()
    return replacement


def build_swap_dictionary(pairs: List[Tuple[str, str]]) -> Dict[str, str]:
    """
    Builds a two-way identity-term mapping.
    """
    swap_dict = {}

    for a, b in pairs:
        a_low = a.lower()
        b_low = b.lower()
        swap_dict[a_low] = b_low
        swap_dict[b_low] = a_low

    return swap_dict


IDENTITY_SWAP_DICT = build_swap_dictionary(IDENTITY_SWAP_PAIRS)

# Single regex pattern for all identity terms.
# This avoids sequential pair-by-pair replacement and prevents chain swaps.
IDENTITY_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(term) for term in sorted(IDENTITY_SWAP_DICT, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE
)


def swap_identity_terms(text: str) -> Tuple[str, bool]:
    """
    Creates an identity-swapped counterfactual by replacing identity terms once.
    Returns:
        cf_text: the counterfactual text
        changed: whether at least one identity term was replaced
    """
    original = str(text)

    changed = False

    def replace_match(match: re.Match) -> str:
        nonlocal changed

        source = match.group(0)
        source_lower = source.lower()

        if source_lower not in IDENTITY_SWAP_DICT:
            return source

        changed = True
        replacement = IDENTITY_SWAP_DICT[source_lower]
        return match_case(replacement, source)

    cf_text = IDENTITY_PATTERN.sub(replace_match, original)

    return cf_text, changed


def add_counterfactual_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds counterfactual text columns:
        cf_text: identity-swapped text
        cf_changed: whether the text actually changed
    """
    df = df.copy()

    results = df["text"].astype(str).apply(swap_identity_terms)

    df["cf_text"] = results.apply(lambda x: x[0])
    df["cf_changed"] = results.apply(lambda x: bool(x[1]))

    return df


civil_train = add_counterfactual_columns(civil_train)
civil_val = add_counterfactual_columns(civil_val)
civil_test = add_counterfactual_columns(civil_test)

print("Counterfactual change rate:")
for name, df in [("train", civil_train), ("val", civil_val), ("test", civil_test)]:
    changed_n = int(df["cf_changed"].sum())
    total_n = len(df)
    print(name, round(changed_n / total_n, 4), f"({changed_n} / {total_n})")

print("\nChange rate by label and identity_any:")
for name, df in [("train", civil_train), ("val", civil_val), ("test", civil_test)]:
    if {"label", "identity_any", "cf_changed"}.issubset(df.columns):
        print(f"\n{name}")
        display(
            df.groupby(["label", "identity_any"])["cf_changed"]
            .agg(["mean", "sum", "count"])
            .reset_index()
        )

example_cols = ["text", "cf_text", "label"]
if "primary_identity" in civil_train.columns:
    example_cols.append("primary_identity")

display(
    civil_train.loc[civil_train["cf_changed"], example_cols]
    .head(8)
)

In [ ]:
# ============================================================
# 6. Group weights for proposed method
# ============================================================

def add_group_weights(df: pd.DataFrame, group_col: str = "wg_group") -> pd.DataFrame:
    df = df.copy()
    counts = df[group_col].value_counts().to_dict()
    raw_weights = df[group_col].map(lambda g: 1.0 / counts[g]).astype(float)
    # Normalise to mean 1 so that loss scale stays stable.
    df["group_weight"] = raw_weights / raw_weights.mean()
    return df

civil_train = add_group_weights(civil_train)
civil_val["group_weight"] = 1.0
civil_test["group_weight"] = 1.0

display(
    civil_train.groupby("wg_group")
    .agg(count=("text", "size"), mean_weight=("group_weight", "mean"))
    .reset_index()
)

In [ ]:
# ============================================================
# 7. Tokenizer and PyTorch datasets
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicityDataset(Dataset):
    def __init__(self, df: pd.DataFrame, use_cf: bool = False):
        self.df = df.reset_index(drop=True).copy()
        self.use_cf = use_cf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        item = {
            "text": row["text"],
            "label": float(row["label"]),
            "group_weight": float(row.get("group_weight", 1.0)),
            "cf_changed": float(row.get("cf_changed", False)),
        }
        if self.use_cf:
            item["cf_text"] = row.get("cf_text", row["text"])
        return item

def make_collate_fn(tokenizer, max_length: int = 192, use_cf: bool = False):
    def collate(batch):
        texts = [b["text"] for b in batch]
        labels = torch.tensor([b["label"] for b in batch], dtype=torch.float32)
        weights = torch.tensor([b["group_weight"] for b in batch], dtype=torch.float32)
        cf_changed = torch.tensor([b["cf_changed"] for b in batch], dtype=torch.float32)

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        out = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": labels,
            "weights": weights,
            "cf_changed": cf_changed,
        }

        if use_cf:
            cf_texts = [b.get("cf_text", b["text"]) for b in batch]
            cf_enc = tokenizer(
                cf_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            out["cf_input_ids"] = cf_enc["input_ids"]
            out["cf_attention_mask"] = cf_enc["attention_mask"]

        return out
    return collate

In [ ]:
# ============================================================
# 8. Metric functions
# ============================================================

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def compute_group_auc_table(eval_df, identity_cols) -> pd.DataFrame:
    """
    Computes ROC-AUC for each identity subgroup.
    AUC is only valid when the subgroup contains both toxic and non-toxic examples.
    """
    rows = []

    for col in identity_cols:
        if col not in eval_df.columns:
            continue

        group_df = eval_df[eval_df[col].astype(int) == 1]

        if len(group_df) == 0:
            continue

        rows.append({
            "group": col,
            "n": len(group_df),
            "positive_n": int(group_df["label"].sum()),
            "negative_n": int((1 - group_df["label"]).sum()),
            "auc": safe_auc(group_df["label"], group_df["prob"]),
        })

    available_identity_cols = [c for c in identity_cols if c in eval_df.columns]

    if len(available_identity_cols) > 0:
        no_identity_df = eval_df[eval_df[available_identity_cols].sum(axis=1) == 0]

        rows.append({
            "group": "no_identity",
            "n": len(no_identity_df),
            "positive_n": int(no_identity_df["label"].sum()),
            "negative_n": int((1 - no_identity_df["label"]).sum()),
            "auc": safe_auc(no_identity_df["label"], no_identity_df["prob"]),
        })

    return pd.DataFrame(rows)


def compute_worst_group_auc(eval_df, identity_cols) -> float:
    """
    Worst-group AUC = minimum valid ROC-AUC across identity subgroups.
    Groups with only one class are ignored because ROC-AUC is undefined there.
    """
    group_auc_df = compute_group_auc_table(eval_df, identity_cols)
    valid_auc_df = group_auc_df.dropna(subset=["auc"])

    if len(valid_auc_df) == 0:
        return np.nan

    return float(valid_auc_df["auc"].min())

def find_best_threshold(y_true, y_prob, objective: str = "macro_f1") -> float:
    thresholds = np.linspace(0.05, 0.95, 91)
    best_t, best_score = 0.5, -1.0

    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        if objective == "macro_f1":
            score = f1_score(y_true, pred, average="macro", zero_division=0)
        else:
            score = accuracy_score(y_true, pred)
        if score > best_score:
            best_score = score
            best_t = float(t)

    return best_t

def classification_metrics(y_true, y_prob, threshold: float) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "roc_auc": safe_auc(y_true, y_prob),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "threshold": threshold,
    }

def fpr(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    neg = y_true == 0
    if neg.sum() == 0:
        return np.nan
    return float((y_pred[neg] == 1).mean())

def compute_bias_metrics(
    df: pd.DataFrame,
    y_prob: np.ndarray,
    threshold: float,
    identity_cols: List[str] = IDENTITY_COLS,
) -> Dict[str, float]:
    eval_df = df.copy().reset_index(drop=True)
    eval_df["prob"] = y_prob
    eval_df["pred"] = (eval_df["prob"] >= threshold).astype(int)

    metrics = {}

    # Worst-group accuracy over label x identity_any groups.
    group_accs = []
    for g, gdf in eval_df.groupby("wg_group"):
        if len(gdf) > 0:
            acc = accuracy_score(gdf["label"], gdf["pred"])
            metrics[f"acc_group_{g}"] = acc
            group_accs.append(acc)
    metrics["worst_group_acc"] = float(np.nanmin(group_accs)) if group_accs else np.nan
    # Worst-group AUC over identity subgroups.
    # This is different from worst_group_acc because AUC must be calculated
    # on groups that contain both positive and negative examples.
    metrics["worst_group_auc"] = compute_worst_group_auc(eval_df, identity_cols)

    # Borkan-style unintended-bias metrics.
    subgroup_aucs = []
    bpsn_aucs = []
    bnsp_aucs = []
    fpr_gaps = []

    for col in identity_cols:
        if col not in eval_df.columns:
            continue

        subgroup = eval_df[col].astype(int) == 1
        background = ~subgroup

        y = eval_df["label"].values
        p = eval_df["prob"].values

        subgroup_auc = safe_auc(y[subgroup], p[subgroup]) if subgroup.sum() > 1 else np.nan

        # BPSN: background positive + subgroup negative.
        bpsn_mask = (background & (eval_df["label"] == 1)) | (subgroup & (eval_df["label"] == 0))
        bpsn_auc = safe_auc(y[bpsn_mask], p[bpsn_mask]) if bpsn_mask.sum() > 1 else np.nan

        # BNSP: background negative + subgroup positive.
        bnsp_mask = (background & (eval_df["label"] == 0)) | (subgroup & (eval_df["label"] == 1))
        bnsp_auc = safe_auc(y[bnsp_mask], p[bnsp_mask]) if bnsp_mask.sum() > 1 else np.nan

        fpr_sub = fpr(eval_df.loc[subgroup, "label"], eval_df.loc[subgroup, "pred"]) if subgroup.sum() > 0 else np.nan
        fpr_bg = fpr(eval_df.loc[background, "label"], eval_df.loc[background, "pred"]) if background.sum() > 0 else np.nan
        fpr_gap = fpr_sub - fpr_bg if not (np.isnan(fpr_sub) or np.isnan(fpr_bg)) else np.nan

        metrics[f"{col}_subgroup_auc"] = subgroup_auc
        metrics[f"{col}_bpsn_auc"] = bpsn_auc
        metrics[f"{col}_bnsp_auc"] = bnsp_auc
        metrics[f"{col}_fpr_gap"] = fpr_gap

        subgroup_aucs.append(subgroup_auc)
        bpsn_aucs.append(bpsn_auc)
        bnsp_aucs.append(bnsp_auc)
        fpr_gaps.append(abs(fpr_gap) if not np.isnan(fpr_gap) else np.nan)

    metrics["mean_subgroup_auc"] = float(np.nanmean(subgroup_aucs)) if len(subgroup_aucs) else np.nan
    metrics["mean_bpsn_auc"] = float(np.nanmean(bpsn_aucs)) if len(bpsn_aucs) else np.nan
    metrics["mean_bnsp_auc"] = float(np.nanmean(bnsp_aucs)) if len(bnsp_aucs) else np.nan
    metrics["max_abs_fpr_gap"] = float(np.nanmax(fpr_gaps)) if len(fpr_gaps) else np.nan

    return metrics

def evaluate_predictions(
    df: pd.DataFrame,
    y_prob: np.ndarray,
    threshold: float,
    identity_cols: List[str] = IDENTITY_COLS,
) -> Dict[str, float]:
    out = classification_metrics(df["label"].values, y_prob, threshold)
    out.update(compute_bias_metrics(df, y_prob, threshold, identity_cols))
    return out

In [ ]:
# ============================================================
# 9. Training and prediction functions
# ============================================================

def prepare_training_df(method: str, train_df: pd.DataFrame) -> Tuple[pd.DataFrame, bool]:
    """
    Returns (training_dataframe, use_counterfactual_pairing).
    """
    method = method.lower()

    if method == "erm":
        df = train_df.copy()
        return df, False

    if method == "cda":
        original = train_df.copy()
        changed = train_df[train_df["cf_changed"]].copy()
        cf_aug = changed.copy()
        cf_aug["text"] = cf_aug["cf_text"]
        df = pd.concat([original, cf_aug], ignore_index=True)
        df["group_weight"] = 1.0
        return df.sample(frac=1, random_state=SEED).reset_index(drop=True), False

    if method in ["clp", "proposed"]:
        df = train_df.copy()
        if method == "clp":
            df["group_weight"] = 1.0
        return df, True

    raise ValueError(f"Unknown method: {method}")

def create_model() -> torch.nn.Module:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
    )
    return model

@torch.no_grad()
def predict_proba(model, df: pd.DataFrame, batch_size: int = 64) -> np.ndarray:
    model.eval()

    ds = ToxicityDataset(df, use_cf=False)

    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate_fn(tokenizer, MAX_LENGTH, use_cf=False),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY if DEVICE.type == "cuda" else False,
        persistent_workers=NUM_WORKERS,
    )

    probs = []

    for batch in tqdm(dl, desc="Predicting", leave=False):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(
            enabled=(USE_AMP and DEVICE.type == "cuda")
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits.squeeze(-1)

        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)

    return np.concatenate(probs)

@torch.no_grad()
def predict_texts(model, texts: List[str], batch_size: int = 64) -> np.ndarray:
    model.eval()
    probs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        logits = model(**enc).logits.squeeze(-1)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(probs)

def train_one_method(method: str, train_df: pd.DataFrame, val_df: pd.DataFrame):
    set_seed(SEED)
    method = method.lower()

    train_used, use_cf = prepare_training_df(method, train_df)

    train_dataset = ToxicityDataset(train_used, use_cf=use_cf)

    train_loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "shuffle": True,
        "collate_fn": make_collate_fn(tokenizer, MAX_LENGTH, use_cf=use_cf),
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY if DEVICE.type == "cuda" else False,
    }

    if NUM_WORKERS > 0:
        train_loader_kwargs["persistent_workers"] = NUM_WORKERS

    train_loader = DataLoader(train_dataset, **train_loader_kwargs)

    model = create_model().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    total_steps = max(1, len(train_loader) * EPOCHS)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    bce = torch.nn.BCEWithLogitsLoss(reduction="none")
    mse = torch.nn.MSELoss(reduction="none")

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and DEVICE.type == "cuda")
    )

    history = []

    # ------------------------------------------------------------
    # Early stopping / best-checkpoint tracking starts here
    # ------------------------------------------------------------
    best_metric_value = -np.inf
    best_epoch = 0
    best_state_dict = None
    best_threshold = 0.5
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_losses = []

        loop = tqdm(
            train_loader,
            desc=f"{method.upper()} epoch {epoch}/{EPOCHS}"
        )

        for batch in loop:
            optimizer.zero_grad(set_to_none=True)

            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)
            weights = batch["weights"].to(DEVICE, non_blocking=True)
            cf_changed = batch["cf_changed"].to(DEVICE, non_blocking=True)

            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and DEVICE.type == "cuda")
            ):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits.squeeze(-1)

                main_loss_vec = bce(logits, labels)

                if method == "proposed":
                    main_loss = (main_loss_vec * weights).mean()
                else:
                    main_loss = main_loss_vec.mean()

                loss = main_loss
                clp_loss_value = torch.tensor(0.0, device=DEVICE)

                if use_cf:
                    cf_input_ids = batch["cf_input_ids"].to(
                        DEVICE,
                        non_blocking=True
                    )
                    cf_attention_mask = batch["cf_attention_mask"].to(
                        DEVICE,
                        non_blocking=True
                    )

                    cf_logits = model(
                        input_ids=cf_input_ids,
                        attention_mask=cf_attention_mask
                    ).logits.squeeze(-1)

                    cf_bce_vec = bce(cf_logits, labels)

                    if method == "proposed":
                        cf_bce_loss = (cf_bce_vec * weights).mean()
                    else:
                        cf_bce_loss = cf_bce_vec.mean()

                    pair_loss_vec = mse(logits, cf_logits)
                    denom = cf_changed.sum().clamp(min=1.0)
                    clp_loss_value = (pair_loss_vec * cf_changed).sum() / denom

                    loss = main_loss + cf_bce_loss + CLP_LAMBDA * clp_loss_value

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_losses.append(float(loss.detach().cpu()))

            loop.set_postfix(
                loss=np.mean(epoch_losses[-20:]),
                clp=float(clp_loss_value.detach().cpu())
            )

        # ------------------------------------------------------------
        # Validation happens after each epoch
        # ------------------------------------------------------------
        val_prob = predict_proba(model, val_df)
        val_threshold = find_best_threshold(
            val_df["label"].values,
            val_prob
        )
        val_metrics = evaluate_predictions(
            val_df,
            val_prob,
            val_threshold
        )

        row = {
            "method": method,
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)),
            "val_macro_f1": val_metrics["macro_f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "val_worst_group_acc": val_metrics["worst_group_acc"],
            "val_worst_group_auc": val_metrics.get("worst_group_auc", np.nan),
            "val_max_abs_fpr_gap": val_metrics["max_abs_fpr_gap"],
            "val_threshold": val_threshold,
        }

        # ------------------------------------------------------------
        # Early stopping decision happens here
        # ------------------------------------------------------------
        current_metric = row[BEST_MODEL_METRIC]

        if current_metric > best_metric_value + EARLY_STOPPING_MIN_DELTA:
            best_metric_value = current_metric
            best_epoch = epoch
            best_threshold = val_threshold
            epochs_without_improvement = 0

            # Save best model weights on CPU to avoid holding extra GPU memory.
            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            row["is_best_epoch"] = True

        else:
            epochs_without_improvement += 1
            row["is_best_epoch"] = False

        row["best_epoch_so_far"] = best_epoch
        row["best_val_metric_so_far"] = best_metric_value
        row["epochs_without_improvement"] = epochs_without_improvement

        history.append(row)
        print(row)

        if EARLY_STOPPING and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping triggered for {method.upper()} at epoch {epoch}. "
                f"Best epoch = {best_epoch}, "
                f"best {BEST_MODEL_METRIC} = {best_metric_value:.5f}."
            )
            break

    history_df = pd.DataFrame(history)

    # ------------------------------------------------------------
    # Load the best checkpoint before returning the model
    # ------------------------------------------------------------
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        model.to(DEVICE)

    history_df["selected_epoch"] = best_epoch
    history_df["selected_val_metric"] = best_metric_value
    history_df["selected_threshold"] = best_threshold

    print(
        f"Selected {method.upper()} checkpoint: "
        f"epoch {best_epoch}, "
        f"{BEST_MODEL_METRIC} = {best_metric_value:.5f}, "
        f"threshold = {best_threshold:.2f}"
    )

    return model, history_df

In [ ]:
# ============================================================
# 10. Run controlled CivilComments experiments
# ============================================================

all_results = []
all_histories = []
test_predictions = {}
trained_models = {}

for method in METHODS_TO_RUN:
    print("\n" + "=" * 80)
    print("Training method:", method)
    print("=" * 80)

    model, history_df = train_one_method(method, civil_train, civil_val)
    history_df.to_csv(OUTPUT_DIR / f"history_{method}.csv", index=False)
    all_histories.append(history_df)

    # Choose threshold only on validation.
    val_prob = predict_proba(model, civil_val)
    threshold = find_best_threshold(civil_val["label"].values, val_prob, objective="macro_f1")

    test_prob = predict_proba(model, civil_test)
    test_predictions[method] = test_prob

    metrics = evaluate_predictions(civil_test, test_prob, threshold)
    metrics["method"] = method
    metrics["threshold"] = threshold
    metrics["selected_epoch"] = int(history_df["selected_epoch"].iloc[-1])
    metrics["selected_val_metric"] = float(history_df["selected_val_metric"].iloc[-1])
    all_results.append(metrics)

    # Keep models for dialect transfer and counterfactual sensitivity.
    # If memory is tight, save only the last/best model and delete others.
    trained_models[method] = model.to("cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(all_results)
metric_order = [
    "method", "roc_auc", "macro_f1", "accuracy", "precision", "recall",
    "worst_group_acc", "worst_group_auc",
    "mean_subgroup_auc", "mean_bpsn_auc", "mean_bnsp_auc",
    "max_abs_fpr_gap", "threshold"
]
display(results_df[[c for c in metric_order if c in results_df.columns]])

results_df.to_csv(OUTPUT_DIR / "civilcomments_main_results.csv", index=False)
pd.concat(all_histories, ignore_index=True).to_csv(OUTPUT_DIR / "training_histories.csv", index=False)

print("Saved results to:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 11. Plot main CivilComments results
# ============================================================

def plot_metric_bar(df: pd.DataFrame, metric: str, title: str, filename: str):
    if metric not in df.columns:
        print(f"Metric not found: {metric}")
        return

    plot_df = df[["method", metric]].dropna().copy()
    plt.figure(figsize=(8, 4))
    plt.bar(plot_df["method"], plot_df[metric])
    plt.ylabel(metric)
    plt.xlabel("Method")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.tight_layout()

    path = OUTPUT_DIR / filename
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

plot_metric_bar(results_df, "macro_f1", "CivilComments: Macro-F1 by method", "civilcomments_macro_f1.png")
plot_metric_bar(results_df, "worst_group_acc", "CivilComments: Worst-group accuracy by method", "civilcomments_worst_group_acc.png")
plot_metric_bar(results_df, "worst_group_auc", "CivilComments: Worst-group AUC by method", "civilcomments_worst_group_auc.png")
plot_metric_bar(results_df, "max_abs_fpr_gap", "CivilComments: Maximum absolute FPR gap by method", "civilcomments_max_abs_fpr_gap.png")
plot_metric_bar(results_df, "mean_bpsn_auc", "CivilComments: Mean BPSN AUC by method", "civilcomments_mean_bpsn_auc.png")

In [ ]:
import matplotlib.pyplot as plt
history_df = pd.concat(all_histories, ignore_index=True)

for method in history_df["method"].unique():
    temp = history_df[history_df["method"] == method]
    plt.plot(temp["epoch"], temp["train_loss"], marker="o", label=method)

plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training loss by method")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
for method in history_df["method"].unique():
    temp = history_df[history_df["method"] == method]
    plt.plot(temp["epoch"], temp["val_roc_auc"], marker="o", label=method)

plt.xlabel("Epoch")
plt.ylabel("Validation ROC-AUC")
plt.title("Validation ROC-AUC by method")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
for method in history_df["method"].unique():
    temp = history_df[history_df["method"] == method]
    plt.plot(temp["epoch"], temp["val_max_abs_fpr_gap"], marker="o", label=method)

plt.xlabel("Epoch")
plt.ylabel("Validation FPR gap")
plt.title("False-positive-rate gap by method")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# Worst-group AUC comparison
# ============================================================

plt.figure(figsize=(7, 5))

plot_df = results_df.sort_values("worst_group_auc", ascending=False)

plt.bar(plot_df["method"], plot_df["worst_group_auc"])

plt.xlabel("Method")
plt.ylabel("Worst-group AUC")
plt.title("Worst-group performance comparison")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ROC curve comparison on CivilComments test set
# ============================================================

from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(7, 5))

for method, probs in test_predictions.items():
    y_true = civil_test["label"].astype(int).values
    y_prob = np.asarray(probs).astype(float)

    fpr_values, tpr_values, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr_values, tpr_values)

    plt.plot(
        fpr_values,
        tpr_values,
        linewidth=2,
        label=f"{method.upper()} (AUC = {roc_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Random")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("CivilComments test ROC curve comparison")
plt.legend()
plt.grid(True)
plt.tight_layout()

path = OUTPUT_DIR / "civilcomments_roc_curve_comparison.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", path)

## Interpreting the main result table

For the paper, avoid claiming that a method is better only because accuracy improved. A stronger HD-level interpretation should compare:

- **classification performance**: ROC-AUC, macro-F1, accuracy;
- **robustness**: worst-group accuracy;
- **unintended bias**: Subgroup AUC, BPSN AUC, BNSP AUC;
- **thresholded harm proxy**: false-positive-rate gap.

A method is more convincing if it improves or preserves overall performance while also improving worst-group and bias metrics. If GB-CLP reduces the FPR gap but lowers macro-F1, that should be discussed as a fairness-performance trade-off rather than hidden.

In [ ]:
# ============================================================
# 12. Load Davidson hate/offensive-language data for dialect transfer
# ============================================================
# Label mapping in the common Davidson dataset:
#   class = 0 hate speech
#   class = 1 offensive language
#   class = 2 neither
# We convert it to binary:
#   toxic = 1 for hate/offensive
#   toxic = 0 for neither

DAVIDSON_DATASET_NAME = "tdavidson/hate_speech_offensive"

def load_davidson_dataset(n: Optional[int] = None, seed: int = 42) -> pd.DataFrame:
    raw = load_dataset(DAVIDSON_DATASET_NAME, split="train")
    df = raw.to_pandas()

    if "tweet" in df.columns:
        df["text"] = df["tweet"].astype(str)
    elif "text" in df.columns:
        df["text"] = df["text"].astype(str)
    else:
        raise ValueError("Could not find tweet/text column in Davidson dataset.")

    if "class" not in df.columns:
        raise ValueError("Could not find class column in Davidson dataset.")

    df["label"] = (df["class"].astype(int) != 2).astype(int)
    df["identity_any"] = 0
    df["primary_identity"] = "dialect_eval"
    df["wg_group"] = "y" + df["label"].astype(str) + "_dialect"
    df["group_weight"] = 1.0

    df = add_counterfactual_columns(df)

    if n is not None and len(df) > n:
        df = stratified_sample(df, n, seed)

    return df.reset_index(drop=True)

davidson_df = load_davidson_dataset(N_DIALECT, SEED)
print(davidson_df.shape)
display(davidson_df.head())
display(davidson_df["label"].value_counts().rename("count").reset_index())

## Dialect-grouping options

The stronger version of the dialectal-transfer experiment should use dialect probabilities from the Blodgett et al. demographic-dialect model, as planned in the Distinction task. This notebook supports that if you provide a CSV file.

Expected optional CSV format:

- one row per Davidson example after loading, or a mergeable identifier;
- either a `dialect_group` column with values such as `AAE_aligned` and `White_aligned`, or probability columns such as `p_aae` and `p_white`.

If no dialect-probability file is supplied, the notebook uses a small transparent **exploratory lexical proxy**. That proxy is useful for checking the pipeline but should not be described as true dialect identification in the final paper.

In [ ]:
# ============================================================
# 13. Dialect grouping
# ============================================================
# Optional: set DIALECT_CSV_PATH to a CSV produced using a dialect model.
# Example:
#   DIALECT_CSV_PATH = "/content/drive/MyDrive/davidson_dialect_probs.csv"
#
# Supported columns:
#   - dialect_group directly, OR
#   - p_aae and p_white, from which groups are derived.

DIALECT_CSV_PATH = None  # Replace with a CSV path if available.

AAE_MARKERS = {
    "finna", "ion", "ima", "imma", "yall", "y'all", "aint", "ain't",
    "tryna", "gon", "gonna", "nah", "bruh", "fam", "yo", "dat", "dis",
    "dem", "wanna", "gotta", "bout", "nigga", "niggas"
}

def exploratory_dialect_proxy(text: str) -> str:
    tokens = re.findall(r"[a-zA-Z']+", str(text).lower())
    score = sum(tok in AAE_MARKERS for tok in tokens)

    if score >= 2:
        return "AAE_proxy"
    return "Other_proxy"

def attach_dialect_groups(df: pd.DataFrame, csv_path: Optional[str] = None) -> Tuple[pd.DataFrame, str]:
    df = df.copy()

    if csv_path is not None and os.path.exists(csv_path):
        probs = pd.read_csv(csv_path)

        # If row counts match, attach by position.
        if len(probs) == len(df):
            for col in probs.columns:
                df[col] = probs[col].values
        else:
            # Try a merge if an identifier exists.
            possible_keys = [k for k in ["id", "tweet_id", "index"] if k in df.columns and k in probs.columns]
            if not possible_keys:
                raise ValueError(
                    "Dialect CSV length does not match and no common merge key was found."
                )
            df = df.merge(probs, on=possible_keys[0], how="left")

        if "dialect_group" in df.columns:
            source = "external_dialect_group"
        elif {"p_aae", "p_white"}.issubset(df.columns):
            df["dialect_group"] = np.where(
                df["p_aae"] >= 0.8, "AAE_aligned",
                np.where(df["p_white"] >= 0.8, "White_aligned", "Other")
            )
            source = "external_probabilities"
        else:
            raise ValueError(
                "Dialect CSV must contain either dialect_group or p_aae and p_white columns."
            )

    else:
        df["dialect_group"] = df["text"].apply(exploratory_dialect_proxy)
        source = "exploratory_proxy"

    return df, source

davidson_df, dialect_source = attach_dialect_groups(davidson_df, DIALECT_CSV_PATH)

print("Dialect grouping source:", dialect_source)
display(davidson_df["dialect_group"].value_counts().rename("count").reset_index())

In [ ]:
# ============================================================
# 14. Dialectal transfer evaluation
# ============================================================

def evaluate_dialect_transfer(
    models: Dict[str, torch.nn.Module],
    df: pd.DataFrame,
    thresholds: Dict[str, float],
) -> pd.DataFrame:
    rows = []

    for method, model in models.items():
        print("Dialect transfer:", method)
        model = model.to(DEVICE)
        probs = predict_proba(model, df)
        threshold = thresholds.get(method, 0.5)

        base = classification_metrics(df["label"].values, probs, threshold)
        preds = (probs >= threshold).astype(int)

        row = {"method": method, "dialect_source": dialect_source, **base}

        for group, gdf in df.assign(prob=probs, pred=preds).groupby("dialect_group"):
            if len(gdf) < 10:
                continue
            row[f"{group}_n"] = len(gdf)
            row[f"{group}_fpr"] = fpr(gdf["label"], gdf["pred"])
            row[f"{group}_positive_rate"] = float(gdf["pred"].mean())
            row[f"{group}_auc"] = safe_auc(gdf["label"], gdf["prob"])

        # Generic max-min FPR gap across available dialect groups.
        group_fprs = [
            row[k] for k in row
            if k.endswith("_fpr") and not pd.isna(row[k])
        ]
        row["dialect_fpr_gap_max_min"] = (
            float(np.nanmax(group_fprs) - np.nanmin(group_fprs))
            if len(group_fprs) >= 2 else np.nan
        )

        rows.append(row)

        model = model.to("cpu")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)

# Use the validation-selected thresholds from the CivilComments experiment.
thresholds = {row["method"]: float(row["threshold"]) for _, row in results_df.iterrows()}

dialect_results_df = evaluate_dialect_transfer(trained_models, davidson_df, thresholds)
display(dialect_results_df)

dialect_results_df.to_csv(OUTPUT_DIR / "dialect_transfer_results.csv", index=False)
print("Saved:", OUTPUT_DIR / "dialect_transfer_results.csv")

In [ ]:
# ============================================================
# 15. Plot dialect-transfer FPR gap
# ============================================================

if "dialect_fpr_gap_max_min" in dialect_results_df.columns:
    plot_metric_bar(
        dialect_results_df,
        "dialect_fpr_gap_max_min",
        "Davidson transfer: max-min dialect FPR gap",
        "dialect_transfer_fpr_gap.png"
    )

## Error analysis

The HD task expects critical analysis, not only positive result reporting. The next cells extract:

1. high-confidence false positives;
2. high-confidence false negatives;
3. false positives by identity subgroup;
4. cases where counterfactual identity swaps cause large prediction changes.

These examples help explain *why* the model behaves differently across groups.

In [ ]:
# ============================================================
# 16. Error analysis on CivilComments
# ============================================================

ANALYSIS_METHOD = "proposed" if "proposed" in test_predictions else list(test_predictions.keys())[-1]
analysis_probs = test_predictions[ANALYSIS_METHOD]
analysis_threshold = thresholds[ANALYSIS_METHOD]

analysis_df = civil_test.copy().reset_index(drop=True)
analysis_df["prob"] = analysis_probs
analysis_df["pred"] = (analysis_df["prob"] >= analysis_threshold).astype(int)
analysis_df["error_type"] = np.where(
    (analysis_df["label"] == 0) & (analysis_df["pred"] == 1), "false_positive",
    np.where((analysis_df["label"] == 1) & (analysis_df["pred"] == 0), "false_negative", "correct")
)

fp_df = (
    analysis_df[analysis_df["error_type"] == "false_positive"]
    .sort_values("prob", ascending=False)
    [["text", "label", "pred", "prob", "identity_any", "primary_identity"]]
    .head(20)
)

fn_df = (
    analysis_df[analysis_df["error_type"] == "false_negative"]
    .sort_values("prob", ascending=True)
    [["text", "label", "pred", "prob", "identity_any", "primary_identity"]]
    .head(20)
)

print("Analysis method:", ANALYSIS_METHOD)
print("High-confidence false positives:")
display(fp_df)
print("High-confidence false negatives:")
display(fn_df)

fp_df.to_csv(OUTPUT_DIR / f"error_analysis_false_positives_{ANALYSIS_METHOD}.csv", index=False)
fn_df.to_csv(OUTPUT_DIR / f"error_analysis_false_negatives_{ANALYSIS_METHOD}.csv", index=False)

In [ ]:
# ============================================================
# 17. Error counts by identity subgroup
# ============================================================

error_group_summary = (
    analysis_df.groupby(["primary_identity", "error_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["correct", "false_positive", "false_negative"]:
    if col not in error_group_summary.columns:
        error_group_summary[col] = 0

error_group_summary["total"] = (
    error_group_summary["correct"]
    + error_group_summary["false_positive"]
    + error_group_summary["false_negative"]
)
error_group_summary["fp_rate_within_group"] = (
    error_group_summary["false_positive"] / error_group_summary["total"].replace(0, np.nan)
)
error_group_summary["fn_rate_within_group"] = (
    error_group_summary["false_negative"] / error_group_summary["total"].replace(0, np.nan)
)

display(error_group_summary.sort_values("total", ascending=False))
error_group_summary.to_csv(OUTPUT_DIR / f"error_group_summary_{ANALYSIS_METHOD}.csv", index=False)

In [ ]:
# ============================================================
# 18. Counterfactual sensitivity analysis
# ============================================================

def counterfactual_sensitivity_for_model(model, df: pd.DataFrame) -> pd.DataFrame:
    cf_df = df[df["cf_changed"]].copy().reset_index(drop=True)

    if len(cf_df) == 0:
        return pd.DataFrame()

    model = model.to(DEVICE)
    p_original = predict_texts(model, cf_df["text"].tolist())
    p_cf = predict_texts(model, cf_df["cf_text"].tolist())
    model = model.to("cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    out = cf_df[["text", "cf_text", "label", "primary_identity"]].copy()
    out["p_original"] = p_original
    out["p_counterfactual"] = p_cf
    out["abs_delta"] = np.abs(out["p_original"] - out["p_counterfactual"])
    return out.sort_values("abs_delta", ascending=False).reset_index(drop=True)

cf_summaries = []
for method, model in trained_models.items():
    print("Counterfactual sensitivity:", method)
    cf_df = counterfactual_sensitivity_for_model(model, civil_test)
    if len(cf_df) == 0:
        continue

    cf_df["method"] = method
    cf_df.to_csv(OUTPUT_DIR / f"counterfactual_sensitivity_examples_{method}.csv", index=False)

    cf_summaries.append({
        "method": method,
        "n_cf_examples": len(cf_df),
        "mean_abs_delta": cf_df["abs_delta"].mean(),
        "median_abs_delta": cf_df["abs_delta"].median(),
        "p90_abs_delta": cf_df["abs_delta"].quantile(0.90),
        "max_abs_delta": cf_df["abs_delta"].max(),
    })

    print("Top examples:")
    display(cf_df.head(8))

cf_summary_df = pd.DataFrame(cf_summaries)
display(cf_summary_df)
cf_summary_df.to_csv(OUTPUT_DIR / "counterfactual_sensitivity_summary.csv", index=False)

In [ ]:
# ============================================================
# 19. Save a compact paper-ready summary table
# ============================================================

paper_summary = results_df[[
    c for c in [
        "method", "roc_auc", "macro_f1", "accuracy",
        "worst_group_acc", "mean_subgroup_auc",
        "mean_bpsn_auc", "mean_bnsp_auc", "max_abs_fpr_gap"
    ]
    if c in results_df.columns
]].copy()

# Round for paper tables.
for c in paper_summary.columns:
    if c != "method":
        paper_summary[c] = paper_summary[c].astype(float).round(4)

paper_summary_path = OUTPUT_DIR / "paper_ready_civilcomments_summary.csv"
paper_summary.to_csv(paper_summary_path, index=False)

display(paper_summary)
print("Saved:", paper_summary_path)

if len(cf_summary_df) > 0:
    cf_paper_path = OUTPUT_DIR / "paper_ready_counterfactual_sensitivity.csv"
    cf_summary_df.round(4).to_csv(cf_paper_path, index=False)
    print("Saved:", cf_paper_path)

## Expected output files

After a full run, the `outputs_hd_toxicity_debiasing/` folder should contain:

- `civilcomments_main_results.csv`
- `training_histories.csv`
- `paper_ready_civilcomments_summary.csv`
- `dialect_transfer_results.csv`
- `counterfactual_sensitivity_summary.csv`
- `error_analysis_false_positives_*.csv`
- `error_analysis_false_negatives_*.csv`
- `error_group_summary_*.csv`
- result figures in `.png` format

These files can be used directly for the Results and Critical Analysis sections of the 2.3HD research paper.